### Imports (bibliotecas que vamos usar no projeto inteiro)

##### ============================================================
##### IMPORTAÇÃO DE BIBLIOTECAS
Aqui importamos tudo que vamos usar no projeto inteiro
Fazemos isso no começo para que todas as células abaixo
já tenham acesso a essas ferramentas
##### ============================================================

In [47]:
# Pandas: biblioteca para trabalhar com tabelas de dados (DataFrames)
# É como uma planilha do Excel dentro do Python
# Apelido "pd" para escrever menos
import pandas as pd

# NumPy: biblioteca para cálculos matemáticos com listas de números
# Muito mais rápida que fazer contas com loops normais do Python
# Apelido "np" para escrever menos
import numpy as np

# datetime: módulo nativo do Python para trabalhar com datas
# - datetime: cria objetos de data (ex: 1 de janeiro de 2024)
# - timedelta: representa uma duração (ex: 30 dias)
from datetime import datetime, timedelta

# random: módulo nativo para gerar números e escolhas aleatórias
# Usamos para simular vendas fictícias que pareçam reais
import random

# os: módulo nativo para interagir com o sistema operacional
# Usamos para criar pastas e verificar se arquivos existem
import os

# re: módulo nativo para expressões regulares
# Expressões regulares são padrões para buscar/limpar textos
import re

# json: módulo nativo para ler e escrever arquivos JSON
# JSON é um formato de dados muito usado em APIs e configurações
import json

# matplotlib.pyplot: biblioteca para criar gráficos
# Apelido "plt" para escrever menos
import matplotlib.pyplot as plt

# seaborn: biblioteca que deixa os gráficos mais bonitos
# Funciona em cima do matplotlib, adicionando estilos prontos
# Apelido "sns" para escrever menos
import seaborn as sns

# Mensagem para confirmar que tudo importou sem erro
print("Todas as bibliotecas importadas com sucesso!")



Todas as bibliotecas importadas com sucesso!


In [48]:
# Agora vamos ler o arquivo CSV e fazer a importação com os dados de vendas

df = pd.read_csv('vendas.csv')




###### ============================================================
#### RF02: Inspecionar e Descrever os Dados

Antes de limpar os dados, precisamos entender o que temos:
quantas linhas, quais colunas, quais tipos, onde estão os erros
É como abrir uma planilha pela primeira vez e dar uma olhada geral
###### ============================================================

In [50]:
def inspecionar_dados(df):
    """Exibe informações básicas do DataFrame."""

    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")

    # shape mostra (número de linhas, número de colunas)
    # Esperamos (200, 8) = 200 vendas com 8 informações cada
    print(f"Shape: {df.shape}")

    # list(df.columns) mostra o nome de todas as colunas
    print(f"\nColunas: {list(df.columns)}")

    # dtypes mostra o tipo de dado de cada coluna
    # object = texto, int64 = número inteiro, float64 = número decimal
    print(f"\nTipos de dados:\n{df.dtypes}")

    # isnull().sum() conta quantos valores nulos tem em cada coluna
    # Valores nulos são os None que colocamos propositalmente no RF01
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")

    # head() mostra as 5 primeiras linhas da tabela
    # Útil para ter uma noção visual dos dados
    print(f"\nPrimeiros registros:\n{df.head()}")

    # describe() calcula estatísticas automáticas das colunas numéricas
    # Mostra: contagem, média, desvio padrão, mínimo, máximo, quartis
    print(f"\nEstatísticas descritivas:\n{df.describe()}")

# Chama a função passando nosso dataset bruto
inspecionar_dados(df)
     




=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade         6
preco_unitario    10
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente     produto     categoria    regiao  \
0         1  2024-05-20  Cliente_035       Mouse   Periféricos   Sudeste   
1         2  2024-02-17  Cliente_042     Teclado   Periféricos     Norte   
2         3  2024-05-22  Cliente_022     Monitor  Computadores  Nordeste   
3         4  2024-06-21  Cliente_017  Smartphone     Celular

###### ============================================================
#### RF03 – LIMPAR E TRATAR OS DADOS
 Aqui tratamos todos os problemas que colocamos de propósito:
- Espaços extras nos nomes de produtos
- Datas inválidas ("DATA INVÁLIDA")
- Valores nulos em quantidade e preço
É uma das etapas mais importantes em ciência de dados
###### ============================================================

In [68]:
# Recarrega os dados brutos do CSV para garantir que está limpo
df_bruto = pd.read_csv("vendas.csv")

# Executa a limpeza usando uma cópia do df_bruto
df_limpo, relatorio = limpar_dados(df_bruto.copy())

def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna o DataFrame limpo e um relatório de limpeza.
    """

# Guarda a quantidade inicial de linhas para comparar depois
    n_inicial = len(df)

    # Dicionário vazio que vai armazenar o relatório de limpeza
    relatorio = {}

    # --- PASSO 1: Remover espaços extras em colunas de texto ---
    # select_dtypes(include="object") pega só as colunas de texto
    # Exemplo: " Notebook" vira "Notebook" depois do strip()
    colunas_texto = df.select_dtypes(include="object").columns
    for col in colunas_texto:
        # .str.strip() remove espaços no início e no fim de cada texto
        df[col] = df[col].str.strip()

    # --- PASSO 2: Converter datas e remover as inválidas ---
    # pd.to_datetime tenta transformar o texto em data real
    # errors="coerce" faz com que textos inválidos virem NaT (Not a Time)
    # Exemplo: "2024-03-15" vira uma data real, "DATA INVÁLIDA" vira NaT
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")

    # Conta quantas datas ficaram nulas (eram inválidas)
    n_datas_invalidas = df["data_venda"].isnull().sum()

    # Remove as linhas onde a data é nula (era "DATA INVÁLIDA")
    # dropna remove linhas com valores nulos na coluna especificada
    df = df.dropna(subset=["data_venda"])
    relatorio["datas_invalidas_removidas"] = n_datas_invalidas

    # --- PASSO 3: Remover linhas com quantidade ou preço nulos ---
    # Não faz sentido uma venda sem quantidade ou sem preço
    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)

    # --- PASSO 4: Garantir tipos numéricos corretos ---
    # Após limpeza, forçamos os tipos para garantir que são números
    # astype(int) converte para número inteiro (sem casas decimais)
    # astype(float) converte para número decimal
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    # Calcula estatísticas finais da limpeza
    n_final = len(df)
    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = n_final
    relatorio["registros_removidos_total"] = n_inicial - n_final

    # Exibe o relatório no terminal
    print("\n=== RELATÓRIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"  {chave}: {valor}")

    # Retorna dois valores: o DataFrame limpo e o relatório
    return df, relatorio

# Executa a limpeza usando uma cópia do df_bruto
# .copy() cria uma cópia independente para não alterar o original
df_limpo, relatorio = limpar_dados(df_bruto.copy())


=== RELATÓRIO DE LIMPEZA ===
  datas_invalidas_removidas: 4
  linhas_nulas_removidas: 16
  registros_iniciais: 200
  registros_finais: 180
  registros_removidos_total: 20

=== RELATÓRIO DE LIMPEZA ===
  datas_invalidas_removidas: 4
  linhas_nulas_removidas: 16
  registros_iniciais: 200
  registros_finais: 180
  registros_removidos_total: 20


/tmp/ipykernel_87483/2493430029.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df.select_dtypes(include="object").columns
